
## ETL/Gold/03 - fact_metricas_diarias (merge idempotente)
## Mapea SKs de dims por BK y calcula delta_downloads / es_primer_dia

In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = get_param("catalog", "pf")
S_MODELO = f"{CATALOG}.silver.modelos"
FACT     = f"{CATALOG}.gold.fact_metricas_diarias"

In [0]:
%py 

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW staging_fact AS
    WITH sil AS (
        SELECT *
        FROM {S_MODELO}
        WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM {S_MODELO})
    )
    SELECT
        sil.model_id AS model_id,
        f.fecha_id AS fecha_id,
        dm.modelo_id AS modelo_id,
        sil.org_id AS org_id,
        do2.org_sk AS org_sk,
        do3.task_id AS task_id,
        dl2.libreria_id AS libreria_id,
        dl3.licencia_id AS licencia_id,
        sil.likes AS likes,
        sil.downloads AS downloads
    FROM sil
    JOIN pf.gold.dim_fecha f
         ON f.fecha = sil.ingestion_date
    JOIN pf.gold.dim_modelo_scd2 dm
         ON dm.model_id = sil.model_id AND dm.is_current = TRUE
    JOIN pf.gold.dim_organizacion do2
         ON do2.org_id = sil.org_id
    LEFT JOIN pf.gold.dim_task do3
         ON do3.pipeline_tag = sil.pipeline_tag
    LEFT JOIN pf.gold.dim_libreria dl2
         ON dl2.library_name = sil.library_name
    LEFT JOIN pf.gold.dim_licencia dl3
         ON dl3.license_tag = COALESCE(sil.license_tag, 'sin_licencia')
""")

In [0]:
%py

spark.sql(f"""
    MERGE INTO {FACT} AS t
    USING (
        SELECT
            MD5(CONCAT_WS('|', s.model_id, s.fecha_id, s.modelo_id)) AS row_hash,
            s.*
        FROM staging_fact s
    ) AS s
    ON t.row_hash = s.row_hash
    WHEN MATCHED THEN
        UPDATE SET t.likes = s.likes,
                   t.downloads = s.downloads
    WHEN NOT MATCHED THEN
        INSERT (row_hash, fecha_id, modelo_id, model_id, org_id, org_sk,
                task_id, libreria_id, licencia_id, likes, downloads,
                delta_downloads, es_primer_dia, _createdAt)
        VALUES (s.row_hash, s.fecha_id, s.modelo_id, s.model_id, s.org_id, s.org_sk,
                s.task_id, s.libreria_id, s.licencia_id, s.likes, s.downloads,
                NULL, TRUE, CURRENT_TIMESTAMP())
""")

In [0]:
%py

today_id = spark.sql("SELECT CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS BIGINT) AS d").collect()[0].d
spark.sql(f"""
    UPDATE {FACT} AS t
    SET t.delta_downloads =
        t.downloads - COALESCE((
            SELECT prev.downloads
            FROM {FACT} prev
            WHERE prev.model_id = t.model_id
              AND prev.fecha_id  = CAST(DATE_FORMAT(DATE_SUB(TO_DATE(CAST(t.fecha_id AS STRING), 'yyyyMMdd'), 1), 'yyyyMMdd') AS BIGINT)
            LIMIT 1), t.downloads)
    WHERE t.fecha_id = {today_id}
""")

In [0]:
%py

spark.sql(f"""
    SELECT COUNT(*) AS filas_fact,
           SUM(CASE WHEN task_id IS NULL THEN 1 ELSE 0 END) AS sin_task,
           SUM(CASE WHEN libreria_id IS NULL THEN 1 ELSE 0 END) AS sin_libreria,
           SUM(CASE WHEN licencia_id IS NULL THEN 1 ELSE 0 END) AS sin_licencia
    FROM {FACT}
""").show()

In [0]:
%py

spark.sql(f"SELECT * FROM {FACT} ORDER BY downloads DESC LIMIT 10").show()